# 🧹 Notebook 2 — Text Preprocessing (NLP Pipeline)
**Project:** AI-Driven Citizen Grievance Analysis  
**Input:**  `data/cleaned/grievance_cleaned.csv` ← from Notebook 1  
**Output:** `data/processed/grievance_processed.csv` → used in Notebook 3

### What this notebook does:
- Lowercase conversion
- Remove URLs, special characters, numbers
- Tokenisation (split text into words)
- Stopword removal (English + custom civic words)
- Lemmatisation (reduce words to root form)
- Token count statistics

---

##  Import Libraries & Setup Paths

In [1]:
import re
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
for pkg in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

# ── Folder Paths (relative to notebooks/ folder) ─────────────────────────────
OUTPUT_DIR        = '../data/'
INPUT_FILE        = OUTPUT_DIR + 'cleaned/grievance_cleaned.csv'
OUTPUT_FILE       = OUTPUT_DIR + 'processed/grievance_processed.csv'

print('✅ Libraries ready!')
print(f'📂 Reading from : {INPUT_FILE}')
print(f'📂 Saving to    : {OUTPUT_FILE}')

✅ Libraries ready!
📂 Reading from : ../data/cleaned/grievance_cleaned.csv
📂 Saving to    : ../data/processed/grievance_processed.csv


##  Load Cleaned Data from Notebook 1

In [2]:
df = pd.read_csv(INPUT_FILE)

print(f'Loaded {len(df):,} rows')
print(f'Columns: {df.columns.tolist()}')
df[['Complaint Type', 'combined_text']].head(3)

Loaded 10,000 rows
Columns: ['Unique Key', 'Created Date', 'Closed Date', 'Agency Name', 'Complaint Type', 'Descriptor', 'Location Type', 'Borough', 'Status', 'Resolution Description', 'resolution_hours', 'hour_created', 'day_of_week', 'combined_text']


,Complaint Type,combined_text
0,Noise - Street/Sidewalk,Noise - Street/Sidewalk Loud Music/Party The P...
1,Blocked Driveway,Blocked Driveway No Access The Police Departme...
2,Blocked Driveway,Blocked Driveway No Access The Police Departme...


## Build the Stop Word List

In [3]:
# Standard English stopwords from NLTK
english_stops = set(stopwords.words('english'))

# Extra civic/domain words that carry no useful meaning for our model
CIVIC_EXTRA = {
    'police', 'department', 'responded', 'complaint', 'arrival',
    'condition', 'information', 'available', 'time', 'new', 'york',
    'city', 'nyc', 'request', 'service', 'action', 'fix', 'officer',
    'determined', 'necessary', 'observed', 'evidence', 'violation'
}

STOP_WORDS = english_stops | CIVIC_EXTRA

print(f'Total stopwords   : {len(STOP_WORDS)}')
print(f'English stopwords : {len(english_stops)}')
print(f'Custom civic extra: {len(CIVIC_EXTRA)}')

Total stopwords   : 221
English stopwords : 198
Custom civic extra: 23


##  Define the Preprocessing Function
This is the **core NLP pipeline**. Run this cell to define it, then apply it next cell.

In [4]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Full NLP preprocessing pipeline:
    1. Lowercase
    2. Remove URLs and emails
    3. Remove special characters and numbers
    4. Remove extra spaces
    5. Tokenise
    6. Remove stopwords and short tokens
    7. Lemmatise
    """
    # Step 1: Lowercase everything
    text = text.lower()

    # Step 2: Remove URLs and emails
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', ' ', text)

    # Step 3: Remove anything that is NOT a letter or space
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Step 4: Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 5: Tokenise — split into individual words
    tokens = word_tokenize(text)

    # Step 6: Remove stopwords and words <= 2 characters
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]

    # Step 7: Lemmatise — e.g. 'running' → 'run', 'blocked' → 'block'
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

print('✅ preprocess_text() function defined!')

✅ preprocess_text() function defined!


## Test the Function on a Sample

In [5]:
# Try a real example from the dataset
sample = df['combined_text'].iloc[0]

print('──── BEFORE ────')
print(sample)
print()
print('──── AFTER  ────')
print(preprocess_text(sample))

──── BEFORE ────
Noise - Street/Sidewalk Loud Music/Party The Police Department responded and upon arrival those responsible for the condition were gone.

──── AFTER  ────
noise street sidewalk loud music party upon responsible gone


##  Apply to All 10,000 Rows
⏱️ This takes about 30–60 seconds.

In [6]:
print('Preprocessing all rows... please wait...')

df['clean_text'] = df['combined_text'].apply(preprocess_text)

print(f'✅ Done! Processed {len(df):,} rows.')
df[['combined_text', 'clean_text']].head(3)

Preprocessing all rows... please wait...
✅ Done! Processed 10,000 rows.


,combined_text,clean_text
0,Noise - Street/Sidewalk Loud Music/Party The P...,noise street sidewalk loud music party upon re...
1,Blocked Driveway No Access The Police Departme...,blocked driveway access
2,Blocked Driveway No Access The Police Departme...,blocked driveway access upon responsible gone


##  Token Count Statistics

In [7]:
df['token_count']   = df['clean_text'].apply(lambda x: len(x.split()))
df['unique_tokens'] = df['clean_text'].apply(lambda x: len(set(x.split())))

print('=== Token Count Statistics ===')
print(df['token_count'].describe().round(2))

print('\n=== Sample token_count per row ===')
print(df[['Complaint Type', 'token_count', 'clean_text']].head(5).to_string())

=== Token Count Statistics ===
count    10000.00
mean         5.68
std          2.05
min          2.00
25%          4.00
50%          6.00
75%          7.00
max         41.00
Name: token_count, dtype: float64

=== Sample token_count per row ===
            Complaint Type  token_count                                                    clean_text
0  Noise - Street/Sidewalk            9  noise street sidewalk loud music party upon responsible gone
1         Blocked Driveway            3                                       blocked driveway access
2         Blocked Driveway            6                 blocked driveway access upon responsible gone
3          Illegal Parking            6             illegal parking commercial overnight parking took
4          Illegal Parking            7        illegal parking blocked sidewalk upon responsible gone


## Check for Empty Rows After Cleaning

In [8]:
empty_rows = df[df['clean_text'].str.strip() == '']
print(f'Rows with empty clean_text: {len(empty_rows)}')

# If any exist, remove them
if len(empty_rows) > 0:
    df = df[df['clean_text'].str.strip() != '']
    print(f'Removed empty rows. New total: {len(df):,}')
else:
    print('✅ No empty rows. All good!')

Rows with empty clean_text: 0
✅ No empty rows. All good!


##  Before vs After Comparison Table

In [9]:
comparison = df[['Complaint Type', 'combined_text', 'clean_text', 'token_count']].head(8).copy()

# Truncate long text for display
comparison['combined_text'] = comparison['combined_text'].str[:80] + '...'
comparison['clean_text']    = comparison['clean_text'].str[:80] + '...'

print(comparison.to_string(index=False))

         Complaint Type                                                                       combined_text                                                      clean_text  token_count
Noise - Street/Sidewalk Noise - Street/Sidewalk Loud Music/Party The Police Department responded and upo... noise street sidewalk loud music party upon responsible gone...            9
       Blocked Driveway Blocked Driveway No Access The Police Department responded to the complaint and ...                                      blocked driveway access...            3
       Blocked Driveway Blocked Driveway No Access The Police Department responded and upon arrival thos...                blocked driveway access upon responsible gone...            6
        Illegal Parking Illegal Parking Commercial Overnight Parking The Police Department responded to ...            illegal parking commercial overnight parking took...            6
        Illegal Parking Illegal Parking Blocked Sidewalk The Police Departm

## Save Preprocessed Data

In [10]:
df.to_csv(OUTPUT_FILE, index=False)

print(f'✅ Saved → {OUTPUT_FILE}')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {df.shape[1]}')
print(f'   Key columns added: clean_text, token_count, unique_tokens')
print()
print('👉 Now open Notebook 3: 03_eda_visualizations.ipynb')

✅ Saved → ../data/processed/grievance_processed.csv
   Rows    : 10,000
   Columns : 17
   Key columns added: clean_text, token_count, unique_tokens

👉 Now open Notebook 3: 03_eda_visualizations.ipynb
